In [ ]:
!pip install -r requirements.txt

In [ ]:
# IMPORTS

from __future__ import print_function
import os, sys, time, datetime, json, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import clone_model
from keras.models import Sequential
from keras.layers import Dense, Activation, PReLU
from keras.optimizers import SGD , Adam, RMSprop
import matplotlib.pyplot as plt
from TreasureMaze import TreasureMaze
from GameExperience import GameExperience
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
%matplotlib inline

In [ ]:
#CONFIG 
# Defines the main configuration for the project, including the maze layout, action mapping, and default training settings
# Converts the maze layout into a NumPy array so it can be used by the game and model

CONFIG = {
    "maze_layout": [
        [1., 0., 1., 1., 1., 1., 1., 1.],
        [1., 0., 1., 1., 1., 0., 1., 1.],
        [1., 1., 1., 1., 0., 1., 0., 1.],
        [1., 1., 1., 0., 1., 1., 1., 1.],
        [1., 1., 0., 1., 1., 1., 1., 1.],
        [1., 1., 1., 0., 1., 0., 0., 0.],
        [1., 1., 1., 0., 1., 1., 1., 1.],
        [1., 1., 1., 1., 0., 1., 1., 1.],
    ],
    "actions": {
        "LEFT": 0,
        "UP": 1,
        "RIGHT": 2,
        "DOWN": 3,
    },
    "training": { 
        "n_epoch": 1000,
        "max_memory": 1000,
        "data_size": 32,
        "target_update_freq": 50,
        "train_every": 4,
        "epsilon_start": 1.0, # Essentially the randomness, how often it will choose
        "epsilon_min": 0.05,  # Exploration vs. exploitation
        "epsilon_decay": 0.995,
        "max_steps_per_episode": 256,
    }
}

maze = np.array(CONFIG["maze_layout"], dtype=np.float32)

In [ ]:
# MOVE / ACTION DICT 
# Creates constants for the possible movement actions and builds helper dictionaries/config variables used throughout training

LEFT = CONFIG["actions"]["LEFT"]
UP = CONFIG["actions"]["UP"]
RIGHT = CONFIG["actions"]["RIGHT"]
DOWN = CONFIG["actions"]["DOWN"]

actions_dict = {
    LEFT: "left",
    UP: "up",
    RIGHT: "right",
    DOWN: "down",
}

num_actions = len(actions_dict)

TRAINING_CONFIG = CONFIG["training"].copy()

In [ ]:
# CORE GAMEPLAY FUNCTIONS
# Contains the core gameplay and model helper functions
# show() visualizes the maze and the pirate’s path
# play_game() runs the trained model through the maze from a chosen start point
# completion_check() tests whether the model can solve the maze reliably
# build_model() creates and compiles the neural network used for Q-learning
# format_time() converts elapsed seconds into a more readable time string


# Displays the current maze state as a grayscale image
# Marks visited cells, the pirate’s current position, and the treasure location
# so the user can visually track progress through the maze
def show(qmaze):
    plt.grid('on')
    nrows, ncols = qmaze.maze.shape
    ax = plt.gca()
    ax.set_xticks(np.arange(0.5, nrows, 1))
    ax.set_yticks(np.arange(0.5, ncols, 1))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    canvas = np.copy(qmaze.maze)
    for row,col in qmaze.visited:
        canvas[row,col] = 0.6
    pirate_row, pirate_col, _ = qmaze.state
    canvas[pirate_row, pirate_col] = 0.3   # pirate cell
    canvas[nrows-1, ncols-1] = 0.9 # treasure cell
    img = plt.imshow(canvas, interpolation='none', cmap='gray')
    return img

# Runs the trained model through the maze from a chosen starting cell
# At each step, the model selects the action with the highest predicted Q-value
# Returns True if the treasure is reached, and False if the agent loses or times out
def play_game(model, qmaze, pirate_cell, max_steps=None):
    qmaze.reset(pirate_cell)
    envstate = qmaze.observe()
    steps = 0
    if max_steps is None:
        max_steps = qmaze.maze.size * 4  # safety cutoff

    while steps < max_steps:
        state = np.asarray(envstate, dtype=np.float32)
        if state.ndim == 1:
            state = np.expand_dims(state, axis=0)

        q_values = model(state, training=False).numpy()
        action = np.argmax(q_values[0])

        envstate, reward, game_status = qmaze.act(action)
        steps += 1

        if game_status == 'win':
            return True
        elif game_status == 'lose':
            return False

    return False  # timed out with no result

# Tests whether the trained model can successfully solve the maze from each valid free cell
# Accepts either a raw maze array or an existing TreasureMaze object
# Returns True only if the model reaches the treasure from every tested starting point
def completion_check(model, maze_or_qmaze, max_steps=None):
    # Accept either raw numpy maze or TreasureMaze instance
    if isinstance(maze_or_qmaze, TreasureMaze):
        qmaze = maze_or_qmaze
    else:
        qmaze = TreasureMaze(maze_or_qmaze)

    for cell in qmaze.free_cells:
        if not qmaze.valid_actions(cell):
            continue
        if not play_game(model, qmaze, cell, max_steps=max_steps):
            return False
    return True


# Builds and compiles the neural network used by the agent for deep Q-learning
# The model takes the flattened maze state as input and outputs one Q-value for each possible action
def build_model(maze):
    model = Sequential()
    model.add(Dense(maze.size, input_shape=(maze.size,)))
    model.add(PReLU())
    model.add(Dense(maze.size))
    model.add(PReLU())
    model.add(Dense(num_actions))
    model.compile(optimizer='adam', loss='mse')
    return model

In [ ]:
# Quick test cell that creates a maze instance, makes one sample move, prints the reward, and displays the maze

qmaze = TreasureMaze(maze)
canvas, reward, game_over = qmaze.act(DOWN)
print("reward=", reward)
show(qmaze)

In [ ]:
# TRAIN STEP
# Defines the TensorFlow training step used during learning
# make_train_step() builds a reusable function that performs one forward pass, computes loss, and applies gradient updates to the model

loss_fn = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam()

# Creates a reusable TensorFlow training function for the Q-learning model
# This function performs one training update on a batch of input states and target Q-values
def make_train_step(model):
    # Gets the size of the model's input layer
    # This is the number of values in one maze state
    input_dim = model.input_shape[-1]

    # Gets the size of the model's output layer
    # This is the number of Q-values the model predicts, one for each action
    output_dim = model.output_shape[-1]

    # Wraps the inner training function with TensorFlow graph execution
    # reduce_retracing=True helps avoid rebuilding the graph unnecessarily
    # input_signature defines the expected shapes/types of x and y
    @tf.function(
        reduce_retracing=True,
        input_signature=[
            tf.TensorSpec(shape=[None, input_dim], dtype=tf.float32),
            tf.TensorSpec(shape=[None, output_dim], dtype=tf.float32),
        ],
    )
    def train_step(x, y):
        # Records operations so TensorFlow can compute gradients automatically
        with tf.GradientTape() as tape:
            # Runs the model on the batch of input states in training mode
            q_values = model(x, training=True)

            # Compares the model's predicted Q-values to the target Q-values
            # This produces the loss used to guide learning
            loss = loss_fn(y, q_values)

        # Computes the gradient of the loss with respect to the model's trainable weights.
        grads = tape.gradient(loss, model.trainable_variables)

        # Applies the gradients to update the model weights using the optimizer
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        
        return loss

    return train_step

In [ ]:
# INPUT VALIDATION FUNCTIONS
# Contains validation functions that check user and configuration inputs before training starts
# validate_maze() ensures the maze is valid and usable
# validate_training_params() ensures training values are within acceptable ranges

def validate_maze(maze):
    maze = np.asarray(maze, dtype=np.float32)

    if maze.ndim != 2:
        raise ValueError("Maze must be a 2D array.")

    if maze.shape[0] == 0 or maze.shape[1] == 0:
        raise ValueError("Maze cannot be empty.")

    unique_values = set(np.unique(maze))
    if not unique_values.issubset({0.0, 1.0}):
        raise ValueError("Maze must contain only 0.0 and 1.0 values.")

    if maze[0, 0] != 1.0:
        raise ValueError("The starting cell (0, 0) must be a free cell with value 1.0.")

    if maze[-1, -1] != 1.0:
        raise ValueError("The treasure cell (bottom-right) must be a free cell with value 1.0.")

    if np.count_nonzero(maze == 1.0) < 2:
        raise ValueError("Maze must contain at least two free cells.")

    return maze


def validate_training_params(
    n_epoch,
    max_memory,
    data_size,
    target_update_freq,
    train_every,
    epsilon_start,
    epsilon_min,
    epsilon_decay,
    max_steps_per_episode,
):
    if n_epoch <= 0:
        raise ValueError("n_epoch must be greater than 0.")

    if max_memory <= 0:
        raise ValueError("max_memory must be greater than 0.")

    if data_size <= 0:
        raise ValueError("data_size must be greater than 0.")

    if target_update_freq <= 0:
        raise ValueError("target_update_freq must be greater than 0.")

    if train_every <= 0:
        raise ValueError("train_every must be greater than 0.")

    if max_steps_per_episode <= 0:
        raise ValueError("max_steps_per_episode must be greater than 0.")

    if not (0.0 <= epsilon_start <= 1.0):
        raise ValueError("epsilon_start must be between 0.0 and 1.0.")

    if not (0.0 <= epsilon_min <= 1.0):
        raise ValueError("epsilon_min must be between 0.0 and 1.0.")

    if epsilon_min > epsilon_start:
        raise ValueError("epsilon_min cannot be greater than epsilon_start.")

    if not (0.0 < epsilon_decay <= 1.0):
        raise ValueError("epsilon_decay must be greater than 0.0 and less than or equal to 1.0.")

In [ ]:
# QTRAIN HELPER FUNCTIONS
# There's a lot...
# Contains helper functions used by qtrain()
# initialize_training() prepares the environment, target network, replay memory, and training function
# select_action() chooses either a random action or the model’s best predicted action
# train_on_batch() performs one replay-memory training update
# update_exploration() adjusts epsilon over time
# print_epoch_status() prints readable progress information after each epoch
# resolve_training_params() fills in any missing training arguments from the default config
# run_training_epoch() handles the logic for one full training epoch

def initialize_training(model, maze, max_memory):
    start_time = datetime.datetime.now()
    qmaze = TreasureMaze(maze)

    target_model = clone_model(model)
    target_model.set_weights(model.get_weights())

    experience = GameExperience(model, target_model, max_memory=max_memory)
    train_step = make_train_step(model)

    return start_time, qmaze, target_model, experience, train_step


def select_action(qmaze, experience, envstate, epsilon):
    if np.random.rand() < epsilon:
        valid_actions = qmaze.valid_actions()
        return random.choice(valid_actions)

    q_values = experience.predict(envstate)
    return int(np.argmax(q_values))


def train_on_batch(experience, train_step, data_size):
    inputs, targets = experience.get_data(data_size)
    if inputs is None:
        return 0.0

    x = tf.convert_to_tensor(inputs, dtype=tf.float32)
    y = tf.convert_to_tensor(targets, dtype=tf.float32)

    batch_loss = train_step(x, y)
    return float(batch_loss.numpy())


def update_exploration(epsilon, win_rate, epsilon_decay, epsilon_min):
    if win_rate > 0.9:
        return 0.05
    return max(epsilon * epsilon_decay, epsilon_min)


def print_epoch_status(epoch, n_epoch, loss, n_episodes, win_history, win_rate, start_time):
    dt = datetime.datetime.now() - start_time
    t = format_time(dt.total_seconds())
    print(
        "Epoch: {:03d}/{:d} | Loss: {:.4f} | Episodes: {:d} | "
        "Win count: {:d} | Win rate (of last 32 epochs): {:.3f} | time: {}".format(
            epoch,
            n_epoch - 1,
            loss,
            n_episodes,
            sum(win_history),
            win_rate,
            t,
        )
    )

def format_time(seconds):
 if seconds < 400:
     s = float(seconds)
     return "%.1f seconds" % (s,)
 elif seconds < 4000:
     m = seconds / 60.0
     return "%.2f minutes" % (m,)
 else:
     h = seconds / 3600.0
     return "%.2f hours" % (h,)

def resolve_training_params(
    n_epoch,
    max_memory,
    data_size,
    target_update_freq,
    train_every,
    epsilon_start,
    epsilon_min,
    epsilon_decay,
    max_steps_per_episode,
):
    return {
        "n_epoch": TRAINING_CONFIG["n_epoch"] if n_epoch is None else n_epoch,
        "max_memory": TRAINING_CONFIG["max_memory"] if max_memory is None else max_memory,
        "data_size": TRAINING_CONFIG["data_size"] if data_size is None else data_size,
        "target_update_freq": TRAINING_CONFIG["target_update_freq"] if target_update_freq is None else target_update_freq,
        "train_every": TRAINING_CONFIG["train_every"] if train_every is None else train_every,
        "epsilon_start": TRAINING_CONFIG["epsilon_start"] if epsilon_start is None else epsilon_start,
        "epsilon_min": TRAINING_CONFIG["epsilon_min"] if epsilon_min is None else epsilon_min,
        "epsilon_decay": TRAINING_CONFIG["epsilon_decay"] if epsilon_decay is None else epsilon_decay,
        "max_steps_per_episode": (
            TRAINING_CONFIG["max_steps_per_episode"]
            if max_steps_per_episode is None else max_steps_per_episode
        ),
    }


def run_training_epoch(
    qmaze,
    experience,
    train_step,
    epsilon,
    data_size,
    train_every,
    max_steps_per_episode,
):
    loss = 0.0
    steps_this_epoch = 0
    step_count = 0

    agent_cell = random.choice(qmaze.free_cells)
    qmaze.reset(agent_cell)
    envstate = qmaze.observe()
    game_over = False
    game_status = "not_over"

    while not game_over and step_count < max_steps_per_episode:
        previous_envstate = envstate

        action = select_action(qmaze, experience, previous_envstate, epsilon)

        envstate, reward, game_status = qmaze.act(action)
        game_over = (game_status != "not_over")

        # Stores the experience in replay memory as:
        # previous state, chosen action, reward, next state, and terminal flag
        episode = [previous_envstate, action, reward, envstate, game_over]
        experience.remember(episode)

        # Trains the model only when replay memory has enough samples
        # and when the current step matches the configured training frequency
        if len(experience.memory) >= data_size and step_count % train_every == 0:
            loss += train_on_batch(experience, train_step, data_size)

        steps_this_epoch += 1
        step_count += 1

    did_win = 1 if game_status == "win" else 0
    return loss, steps_this_epoch, did_win

In [ ]:
# MAIN TRAINING FUNCTION
# Main training function for the project
# qtrain() runs the reinforcement learning loop across many epochs, stores experiences, updates the model from replay memory, tracks progress, and returns a history dictionary for later analysis or visualization

# Runs the full deep Q-learning training process for the Treasure Hunt agent
# It prepares the training settings, validates inputs, trains the model over many epochs,
# tracks performance over time, and returns a history dictionary for later analysis or plotting
def qtrain(
    model,
    maze,
    n_epoch=None,
    max_memory=None,
    data_size=None,
    target_update_freq=None,
    train_every=None,
    epsilon_start=None,
    epsilon_min=None,
    epsilon_decay=None,
    max_steps_per_episode=None,
    progress_callback=None,
    verbose=True,
):
    # Fills in any missing training arguments using the default configuration
    params = resolve_training_params(
        n_epoch,
        max_memory,
        data_size,
        target_update_freq,
        train_every,
        epsilon_start,
        epsilon_min,
        epsilon_decay,
        max_steps_per_episode,
    )

    # Stores the final training values that will be used in this run
    n_epoch = params["n_epoch"]
    max_memory = params["max_memory"]
    data_size = params["data_size"]
    target_update_freq = params["target_update_freq"]
    train_every = params["train_every"]
    epsilon_start = params["epsilon_start"]
    epsilon_min = params["epsilon_min"]
    epsilon_decay = params["epsilon_decay"]
    max_steps_per_episode = params["max_steps_per_episode"]

    # Validates the maze and training settings before learning begins
    maze = validate_maze(maze)
    validate_training_params(
        n_epoch,
        max_memory,
        data_size,
        target_update_freq,
        train_every,
        epsilon_start,
        epsilon_min,
        epsilon_decay,
        max_steps_per_episode,
    )

    # Sets the starting exploration rate for epsilon-greedy action selection
    epsilon = epsilon_start

    # Initializes the maze environment, target network, replay memory, and training step function
    start_time, qmaze, target_model, experience, train_step = initialize_training(
        model, maze, max_memory
    )

    # Stores the result of each epoch so recent win rate can be calculated
    win_history = []

    # Defines the size of the rolling window used to calculate recent win rate
    hsize = qmaze.maze.size // 2

    # Stores training metrics so they can be graphed or displayed late
    history = {
        "epoch": [],
        "loss": [],
        "steps": [],
        "total_wins": [],
        "recent_win_rate": [],
        "elapsed_seconds": [],
    }

    # Main training loop: runs one full training epoch at a time
    for epoch in range(n_epoch):
        loss, steps_this_epoch, did_win = run_training_epoch(
            qmaze,
            experience,
            train_step,
            epsilon,
            data_size,
            train_every,
            max_steps_per_episode,
        )

        win_history.append(did_win)

        # Periodically copies the online model weights into the target model for more stable learning
        if (epoch + 1) % target_update_freq == 0:
            target_model.set_weights(model.get_weights())

        # Calculates the recent rolling win rate based on the most recent training results
        recent_results = win_history[-hsize:]
        win_rate = (
            sum(recent_results) / len(recent_results)
            if len(recent_results) > 0
            else 0.0
        )

        elapsed_seconds = (datetime.datetime.now() - start_time).total_seconds()
        total_wins = sum(win_history)

        history["epoch"].append(epoch)
        history["loss"].append(loss)
        history["steps"].append(steps_this_epoch)
        history["total_wins"].append(total_wins)
        history["recent_win_rate"].append(win_rate)
        history["elapsed_seconds"].append(elapsed_seconds)

        if verbose:
            print_epoch_status(
                epoch,
                n_epoch,
                loss,
                steps_this_epoch,
                win_history,
                win_rate,
                start_time,
            )

        # Sends live progress updates to the dashboard UI if a callback function was provided
        if progress_callback is not None:
            progress_callback({
                "epoch": epoch + 1,
                "n_epoch": n_epoch,
                "loss": loss,
                "steps": steps_this_epoch,
                "total_wins": total_wins,
                "recent_win_rate": win_rate,
                "elapsed_seconds": elapsed_seconds,
            })

        # Updates epsilon so exploration gradually decreases over time
        epsilon = update_exploration(
            epsilon, win_rate, epsilon_decay, epsilon_min
        )

        # Stops training early if the model appears to have fully solved the maze
        if win_rate >= 0.999 and completion_check(model, maze, max_steps=max_steps_per_episode):
            if verbose:
                print(f"Reached 100% win rate at epoch {epoch}")
            break

    total_time = format_time((datetime.datetime.now() - start_time).total_seconds())
    if verbose:
        print("Training complete in:", total_time)

    return history


In [ ]:
# DISPLAY HELPER FUNCTIONS
# Contains helper functions for the notebook dashboard and result display
# plot_training_history() graphs loss, win rate, and steps per epoch
# build_start_options() creates dropdown options for choosing a starting cell
# render_summary_cards() formats final training results into a simple dashboard-style summary
# run_demo_view() runs a trained model from a selected start point and visualizes the result

def plot_training_history(history):
    epochs = history["epoch"]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(epochs, history["loss"])
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].plot(epochs, history["recent_win_rate"])
    axes[1].set_title("Recent Win Rate")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Win Rate")

    axes[2].plot(epochs, history["steps"])
    axes[2].set_title("Steps per Epoch")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Steps")

    plt.tight_layout()
    plt.show()


def build_start_options(maze):
    qmaze = TreasureMaze(maze)
    options = []
    for cell in qmaze.free_cells:
        options.append((str(cell), cell))
    return options


def render_summary_cards(history):
    last_loss = history["loss"][-1]
    last_steps = history["steps"][-1]
    last_wins = history["total_wins"][-1]
    last_win_rate = history["recent_win_rate"][-1]
    elapsed = format_time(history["elapsed_seconds"][-1])

# I hate html.
    return f"""
    <div style="display:flex; gap:12px; flex-wrap:wrap; margin:10px 0;">
        <div style="padding:12px 16px; border:1px solid #ccc; border-radius:12px; min-width:140px;">
            <b>Total Wins</b><br>{last_wins}
        </div>
        <div style="padding:12px 16px; border:1px solid #ccc; border-radius:12px; min-width:140px;">
            <b>Recent Win Rate</b><br>{last_win_rate:.3f}
        </div>
        <div style="padding:12px 16px; border:1px solid #ccc; border-radius:12px; min-width:140px;">
            <b>Last Epoch Steps</b><br>{last_steps}
        </div>
        <div style="padding:12px 16px; border:1px solid #ccc; border-radius:12px; min-width:140px;">
            <b>Last Loss</b><br>{last_loss:.4f}
        </div>
        <div style="padding:12px 16px; border:1px solid #ccc; border-radius:12px; min-width:140px;">
            <b>Total Time</b><br>{elapsed}
        </div>
    </div>
    """


def run_demo_view(model, maze, start_cell, max_steps):
    qmaze = TreasureMaze(maze)
    won = play_game(model, qmaze, start_cell, max_steps=max_steps)
    show(qmaze)
    plt.title(f"Demo Start: {start_cell} | Result: {'Treasure found' if won else 'No treasure'}")
    plt.show()
    return won

In [ ]:
# DASHBOARD CONFIG AND FUNCTIONALITY
# Builds the interactive dashboard for the notebook
# Creates sliders and buttons for training settings, shows live status updates, runs training through the UI, displays charts, and allows the user to test the trained agent from a selected start position

dashboard_model = None
dashboard_history = None

title = widgets.HTML("<h2 style='margin:0;'>Treasure Hunt Trainer</h2><p style='margin:4px 0 12px 0;'>Train, review results, and demo the learned agent.</p>")

epoch_slider = widgets.IntSlider(
    value=TRAINING_CONFIG["n_epoch"], min=100, max=5000, step=100, description="Epochs:"
)
memory_slider = widgets.IntSlider(
    value=TRAINING_CONFIG["max_memory"], min=100, max=5000, step=100, description="Memory:"
)
batch_slider = widgets.IntSlider(
    value=TRAINING_CONFIG["data_size"], min=4, max=128, step=4, description="Batch:"
)
target_slider = widgets.IntSlider(
    value=TRAINING_CONFIG["target_update_freq"], min=1, max=100, step=1, description="Target:"
)
train_every_slider = widgets.IntSlider(
    value=TRAINING_CONFIG["train_every"], min=1, max=10, step=1, description="Train Every:"
)
epsilon_start_slider = widgets.FloatSlider(
    value=TRAINING_CONFIG["epsilon_start"], min=0.1, max=1.0, step=0.01, description="Epsilon:"
)
epsilon_min_slider = widgets.FloatSlider(
    value=TRAINING_CONFIG["epsilon_min"], min=0.0, max=0.2, step=0.01, description="Min Eps:"
)
epsilon_decay_slider = widgets.FloatSlider(
    value=TRAINING_CONFIG["epsilon_decay"], min=0.90, max=0.999, step=0.001, description="Decay:"
)
max_steps_slider = widgets.IntSlider(
    value=TRAINING_CONFIG["max_steps_per_episode"], min=32, max=512, step=16, description="Max Steps:"
)

run_button = widgets.Button(description="Run Training", button_style="success")
demo_button = widgets.Button(description="Run Demo", button_style="info")

start_dropdown = widgets.Dropdown(
    options=build_start_options(maze),
    value=(0, 0),
    description="Start Cell:"
)

status_html = widgets.HTML("<b>Status:</b> Ready")
summary_html = widgets.HTML("")
training_output = widgets.Output()
charts_output = widgets.Output()
demo_output = widgets.Output()

controls_left = widgets.VBox([
    epoch_slider,
    memory_slider,
    batch_slider,
    target_slider,
    train_every_slider,
])

controls_right = widgets.VBox([
    epsilon_start_slider,
    epsilon_min_slider,
    epsilon_decay_slider,
    max_steps_slider,
    start_dropdown,
])

controls = widgets.HBox([controls_left, controls_right])
buttons = widgets.HBox([run_button, demo_button])


def update_status(info):
    status_html.value = (
        f"<b>Status:</b> Epoch {info['epoch']}/{info['n_epoch']} "
        f"| Loss: {info['loss']:.4f} "
        f"| Steps: {info['steps']} "
        f"| Wins: {info['total_wins']} "
        f"| Recent Win Rate: {info['recent_win_rate']:.3f} "
        f"| Time: {format_time(info['elapsed_seconds'])}"
    )


def on_run_clicked(_):
    global dashboard_model, dashboard_history

    with training_output:
        clear_output()
        print("Training started...")

    with charts_output:
        clear_output()

    with demo_output:
        clear_output()

    summary_html.value = ""
    status_html.value = "<b>Status:</b> Training..."

    dashboard_model = build_model(maze)
    dashboard_history = qtrain(
        dashboard_model,
        maze,
        n_epoch=epoch_slider.value,
        max_memory=memory_slider.value,
        data_size=batch_slider.value,
        target_update_freq=target_slider.value,
        train_every=train_every_slider.value,
        epsilon_start=epsilon_start_slider.value,
        epsilon_min=epsilon_min_slider.value,
        epsilon_decay=epsilon_decay_slider.value,
        max_steps_per_episode=max_steps_slider.value,
        progress_callback=update_status,
        verbose=False,
    )

    summary_html.value = render_summary_cards(dashboard_history)

    with training_output:
        clear_output()
        print("Training complete.")

    with charts_output:
        clear_output()
        plot_training_history(dashboard_history)

    status_html.value = "<b>Status:</b> Training complete"


def on_demo_clicked(_):
    with demo_output:
        clear_output()

        if dashboard_model is None:
            print("Train a model first.")
            return

        won = run_demo_view(
            dashboard_model,
            maze,
            start_dropdown.value,
            max_steps_slider.value,
        )
        print("Result:", "Treasure found" if won else "No treasure / timed out")


run_button.on_click(on_run_clicked)
demo_button.on_click(on_demo_clicked)

display(
    title,
    controls,
    buttons,
    status_html,
    summary_html,
    widgets.HTML("<h3>Training Output</h3>"),
    training_output,
    widgets.HTML("<h3>Results</h3>"),
    charts_output,
    widgets.HTML("<h3>Demo</h3>"),
    demo_output,
)

In [ ]:
# Displays the maze on its own so the user can quickly view the layout before training or testing

qmaze = TreasureMaze(maze)
show(qmaze)

In [ ]:
# Manually creates the model and starts training using the current maze and training configuration
# This is the direct non-dashboard way to run the program

model = build_model(maze)
qtrain(model, maze)

In [ ]:
# Runs a completion check after training to see whether the model can solve the maze reliably, then displays the maze

completion_check(model, qmaze)
show(qmaze)

In [ ]:
# Runs the trained model from a chosen starting point and displays the final path through the maze
# Useful as a quick demonstration of the trained agent’s behavior

pirate_start = (0, 0)
play_game(model, qmaze, pirate_start)
show(qmaze)